## Imports

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from nano_llm.evaluation import compute_accuracy

## Neural Network

In [ ]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),
            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

## Data

In [ ]:
# Train
train_data = torch.tensor(
    [[-1.2, 3.1], [-0.9, 2.9], [-0.5, 2.6], [2.3, -1.1], [2.7, -1.5]]
)

train_labels = torch.tensor([0, 0, 0, 1, 1])

# Test
test_data = torch.tensor(
    [
        [-0.8, 2.8],
        [2.6, -1.6],
    ]
)

test_labels = torch.tensor([0, 1])

## Datasets

In [ ]:
class ToyDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]


train_ds = ToyDataset(train_data, train_labels)
test_ds = ToyDataset(test_data, test_labels)

## DataLoaders

In [ ]:
torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds, batch_size=2, shuffle=True, num_workers=0, drop_last=True
)


test_loader = DataLoader(dataset=test_ds, batch_size=2, shuffle=False, num_workers=0)

## Training Loop

In [ ]:
torch.manual_seed(123)

model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

EPOCH = 3

for epoch in range(EPOCH):
    model.train()  # Set model in training mode

    # Train
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)

        loss = F.cross_entropy(logits, labels)  # Loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(
            f"Epoch: {epoch + 1:03d}/{EPOCH:03d}"
            f" | Batch {batch_idx + 1:03d}/{len(train_loader):03d}"
            f" | Train/Val Loss: {loss:.2f}"
        )

    model.eval()  # Set model in evaluation mode
    # Optional model evaluation

## Compute accuracy

In [ ]:
print(compute_accuracy(model, train_loader))